In [0]:
from pyspark.sql import functions as F

silver_fx_df = spark.table("fraud_detection.silver.transactions_fx_enriched")
blocklist_df = spark.table("fraud_detection.bronze.ip_blocklist_raw")

silver_with_risk = silver_fx_df.join(
    blocklist_df,
    on="ip_address",
    how="left"
).withColumn(
    "is_blocklisted",
    F.when(F.col("blacklist_count").isNotNull(), True).otherwise(False)
).withColumn(
    "blacklist_count",
    F.coalesce(F.col("blacklist_count"), F.lit(0))
)

print(f"Row count: {silver_with_risk.count()}")

blocklisted_summary = silver_with_risk.groupBy("is_blocklisted").count()
display(blocklisted_summary)

In [0]:
display(
    silver_with_risk
    .groupBy("is_blocklisted")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum("is_fraudulent").alias("fraud_count"),
        (F.sum("is_fraudulent") / F.count("*")).alias("fraud_rate")
    )
)

In [0]:
silver_with_risk.write.mode("overwrite").saveAsTable("fraud_detection.silver.transactions_risk_flagged")
display(spark.sql("SELECT COUNT(*) FROM fraud_detection.silver.transactions_risk_flagged"))